In [1]:
# using GNNs to build network graph from nmap output

import xml.etree.ElementTree as ET  # ElementTree library is used to parse XML data
import networkx  # turns ElementTree object to networkx object
import torch
# from torch_geometric.utils.convert import from_networkx

# Read the XML `nmap` Output
- Parse `.xml` into nodes, edges, attributes.

https://www.youtube.com/watch?v=5SlemSWGD1g

Note:
What I have now is not universal, I will need our true `nmap` scans to make this one-size fits all.

In [52]:
tree = ET.parse("data/nmap_output.xml")
root = tree.getroot()  # tag that envelopes everything
root.attrib  # length of 6 (scaninfo, host1, host2, host3, host4, runstats)

{'scanner': 'nmap',
 'args': 'nmap -sV -oX scan.xml 192.168.1.0/29',
 'start': '1733764800',
 'version': '7.94',
 'xmloutputversion': '1.05'}

In [ ]:
# loop over root children and their sub attributes
# find each HOST element
network = {}
for child in root: 
    network_config = {}

    # skip over none host elements
    if child.tag != "host":
        continue

    # print("CHILD:", child)
    # pull all IP hosts found (up/down)
    addr = child.find("address").attrib["addr"]  # might not be universal
    status = child.find("status").attrib["state"]

    if status != "up":
        network_config["os"] = None
        network_config["state"] = None
        network_config["hostname"] = None
        network_config["ports"] = None
    else: 
        # find IP hostname
        hostname_root = child.find("hostnames")
        hostname = hostname_root.find("hostname").attrib["name"]

        # find IP OS
        os_config = {}
        os_root = child.find("os")
        if os_root is not None:
            os = os_root.find("osmatch")
            network_config["os"] = os.attrib
        else: 
            network_config["os"] = None

        network_config["state"] = status 
        network_config["hostname"] = hostname

        # find IP open ports  (might need to clean this up to make it more universal)
        port_lst = []
        port_root = child.find("ports")
        port_info = port_root.find("port")
        for port in port_root:
            for val in port:
                if val.attrib["state"] == "open":
                    port.attrib |= val.attrib
                else:
                    continue
            port_lst.append(port.attrib)

        network_config["ports"] = port_lst

    # add the host into the dictionary
    network[addr] = network_config



print(network)


CHILD: <Element 'host' at 0x000002AE0C9D27F0>
root 1
{'protocol': 'tcp', 'portid': '22', 'state': 'open', 'reason': 'syn-ack', 'name': 'ssh', 'product': 'OpenSSH', 'version': '8.4p1'}
val {'state': 'open', 'reason': 'syn-ack'}
val {'name': 'ssh', 'product': 'OpenSSH', 'version': '8.4p1'}
[{'protocol': 'tcp', 'portid': '22', 'state': 'open', 'reason': 'syn-ack', 'name': 'ssh', 'product': 'OpenSSH', 'version': '8.4p1'}]
root 2
{'protocol': 'tcp', 'portid': '80', 'state': 'open', 'reason': 'syn-ack', 'name': 'http', 'product': 'Apache httpd', 'version': '2.4.41'}
val {'state': 'open', 'reason': 'syn-ack'}
val {'name': 'http', 'product': 'Apache httpd', 'version': '2.4.41'}
[{'protocol': 'tcp', 'portid': '22', 'state': 'open', 'reason': 'syn-ack', 'name': 'ssh', 'product': 'OpenSSH', 'version': '8.4p1'}, {'protocol': 'tcp', 'portid': '80', 'state': 'open', 'reason': 'syn-ack', 'name': 'http', 'product': 'Apache httpd', 'version': '2.4.41'}]
root 3
{'protocol': 'tcp', 'portid': '25', 'state

## Turn ElementTree into `networkx` graph.

## Turn `networkx` graph into GNN graph.

## Output GNN graph object.